In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

from dotenv import dotenv_values

In [3]:
# Load config file agnostic to project or notebook work directory
config_path = Path("config.env")
if not config_path.exists():
    config_path = Path("../../../config.env")
CONFIG = dotenv_values(config_path)
assert CONFIG != {}, 'Configuration is empty - check the filepath.'

# Main data directory
DATADIR = Path(CONFIG['DATADIR'])

# Submodule specific data directory
OUTDIR = DATADIR / '10_analyses' / 'sexual_dimorphism'
OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# Notebook specific constants
SIG_THRESHOLD = 0.05

In [5]:
# Load control single cell AnnData
fp_adata = DATADIR / '9_objects' / 'singlecell' / 'hypomap3d_singlecell_control.h5ad'
adata = sc.read_h5ad(fp_adata)

In [6]:
# Initialise DF scaffold via nested dictionaries
init_df = {}

### Count no. of sex DEGs for each population

In [7]:
# Load full sex DEG table
fp_mv_results_df = OUTDIR / 'L4_multivariate_sex_edgeR_results_full.csv'
mv_results_df = pd.read_csv(fp_mv_results_df, index_col=0)
mv_results_df.head(3)

,logFC,logCPM,F,PValue,FDR,id
Xist,-9.366312,9.111565,693.382285,1.626745e-23,6.650134e-20,Gl4-293
Uty,6.822763,6.542082,398.997189,6.205220e-22,1.268347e-18,Gl4-293
Eif2s3y,7.544865,6.604348,321.044229,2.380016e-16,3.243169e-13,Gl4-293


In [8]:
# Subset to only significant DEGs
sig_only_df = mv_results_df[mv_results_df['FDR'] < SIG_THRESHOLD]
print(f"{sig_only_df.shape[0]} significant DEGs!")

8878 significant DEGs!


In [9]:
# Count no. of DEGs per population
population_n_DEGs = sig_only_df.groupby('id').size()

# Add result to dataframe scaffold
init_df['n_DEGs'] = population_n_DEGs.astype(np.int32)

### Compute pct_Female for each population

In [10]:
sex_df = adata.obs.loc[
    adata.obs['sex_per_sample_inferred'] != 'mixed', # remove mixed samples from calculation
    ['L4_ID', 'sex_per_sample_inferred']
].copy()

# Binaries (1 - Female, 0 - Male)
sex_df['isFemale'] = sex_df['sex_per_sample_inferred'] == 'female'

# Compute pct_Female as mean per population of isFemale
population_pct_female = sex_df.groupby('L4_ID', observed=True)['isFemale'].mean()

# Add result to dataframe scaffold
init_df['pct_Female'] = population_pct_female

### Compute sLISI for each population

In [11]:
from hm3d_utils.statistics import calculate_lisi_scores

In [12]:
# Subset control AnnData to drop `mixed` samples
invalid_mask = adata.obs['sex_per_sample_inferred'] == 'mixed'
subset = adata[~invalid_mask, :]
print(f"{subset.shape[0]} cells remain with definite sex annotation.")

839213 cells remain with definite sex annotation.


In [13]:
# Compute LISI scores between male and female
lisi_scores = calculate_lisi_scores(
    embedding=subset.obsm['X_embed'],
    labels=subset.obs['sex_per_sample_inferred'].cat.codes,
    perplexity=30,
    k_factor=3,
    scaled=True,
    use_gpu=True
)

# Invert score to get sLISI
slisi_scores = 1 - lisi_scores

# Get upper quartile of sLISI for each population
subset_df = subset.obs[['L4_ID']].copy()
subset_df['sLISI'] = slisi_scores

# Add result to dataframe scaffold
init_df['sLISI_Q1'] = subset_df.groupby('L4_ID')['sLISI'].quantile(0.25)
init_df['sLISI_Q2'] = subset_df.groupby('L4_ID')['sLISI'].quantile(0.5)
init_df['sLISI_Q3'] = subset_df.groupby('L4_ID')['sLISI'].quantile(0.75)

/REDACTED-path/.temp/ipykernel_725859/2984311338.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  init_df['sLISI_Q1'] = subset_df.groupby('L4_ID')['sLISI'].quantile(0.25)
/REDACTED-path/.temp/ipykernel_725859/2984311338.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  init_df['sLISI_Q2'] = subset_df.groupby('L4_ID')['sLISI'].quantile(0.5)
/REDACTED-path/.temp/ipykernel_725859/2984311338.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default a

### Compile final metrics DataFrame

In [14]:
df_metrics = pd.DataFrame.from_dict(
    init_df
).reindex(adata.obs['L4_ID'].cat.categories)
df_metrics.index.name = 'celltype_ID'

df_metrics['n_DEGs'] = df_metrics['n_DEGs'].fillna(0).astype(int)

df_metrics['sex_ratio_deviance'] = (0.5 - df_metrics['pct_Female']).abs()

df_metrics.head(2)

,n_DEGs,pct_Female,sLISI_Q1,sLISI_Q2,sLISI_Q3,sex_ratio_deviance
celltype_ID,,,,,,
Gb4-319,3,0.547619,0.018490,0.086078,0.224677,0.047619
Gb4-320,2,0.532297,0.027099,0.095940,0.237342,0.032297


In [17]:
# Save out the metrics table
outfile = OUTDIR / 'L4_sd_metrics.csv'
df_metrics.sort_values('n_DEGs', ascending=False).to_csv(outfile)